# 01 - ESCI Shopping Queries Dataset Exploration

Exploration of the Amazon ESCI (Exact, Substitute, Complement, Irrelevant) Shopping Queries dataset for PRISM.
The dataset has three tables: **sources** (query origin), **products** (catalogue) and **examples** (query-product relevance judgements).

## Summary of findings

**Dataset size**
- `sources`: 130,652 queries, `query_id` is unique (one row per query).
- `products`: 1,814,924 rows across three locales: US 1,215,854 (67.0%), JP 339,059 (18.7%), ES 260,011 (14.3%).
- `examples`: 2,621,288 query-product pairs (about 20 judged products per query on average). The small version has 1,118,011 pairs (about 43% of the large version).

**Label distribution (`esci_label`)**

| Label | Count | Share |
|---|---|---|
| E - Exact | 1,708,158 | 65.2% |
| S - Substitute | 574,313 | 21.9% |
| I - Irrelevant | 263,165 | 10.0% |
| C - Complement | 75,652 | 2.9% |

**Query sources**
- Five source types: `other`, `negations`, `behavioral`, `parse_pattern`, `nlqec`.

| Source        | Count  |
|---------------|-------:|
| other         | 113656 |
| negations     |   6964 |
| parse_pattern |   6083 |
| behavioral    |   3780 |
| nlqec         |    169 |

**Keys and locales**
- `product_id` is unique within the US subset (0 duplicates) but 12,152 IDs repeat in the full table, so the same ID appears in more than one locale.

**Data quality observations**
- Queries contain typos and noise (e.g. `revent 80 cfm`, `!awnmower tires without rims`, `# 10 self-seal envelopes without window`), plus many negation queries ("without ..."), which are hard for lexical matching.
- Product fields are often missing: `product_description`, `product_bullet_point`, `product_brand` and `product_color` contain many NaN values. Some products have only a title.
- Descriptions contain raw HTML (`<p>`, `<br>`, `</br>`) and HTML entities (`&#11088;`, `&amp;`), and color values are inconsistent (`As Picture`, `Color 10`, `With Heater`). These need cleaning before text embedding.
- Query metadata consistency: Validation identified 7 query_id values with inconsistent small_version values in us_examples. All 7 IDs have small_version = 2 unique values, while query is consistent (n_texts = 1) for every ID. Query ID 79706 also has 2 unique split values; the other 6 IDs have consistent splits.

---

In [3]:
import pandas as pd

In [4]:
sources = pd.read_csv("../data/shopping_queries_dataset_sources.csv")
products = pd.read_parquet("../data/shopping_queries_dataset_products.parquet")
examples = pd.read_parquet("../data/shopping_queries_dataset_examples.parquet")

In [42]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [43]:
sources.head(100)

,query_id,source
0,0,other
1,1,negations
2,2,negations
3,3,negations
4,4,behavioral
5,5,negations
6,6,negations
7,7,negations
8,8,other
9,9,other


In [44]:
sources['query_id'].is_unique

True

In [45]:
sources['source'].unique()

<ArrowStringArray>
['other', 'negations', 'behavioral', 'parse_pattern', 'nlqec']
Length: 5, dtype: str

In [46]:
sources['source'].value_counts()

source
other            113656
negations          6964
parse_pattern      6083
behavioral         3780
nlqec               169
Name: count, dtype: int64

In [47]:
sources.shape

(130652, 2)

In [48]:
products[products['product_locale'] == 'us'].head(100)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
167168,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust Bath Fan with Humidity Sensor,NaN,"Virtually silent at less than 0.3 sones\nPrecision engineered with DC brushless motor for extended reliability\nEasily switch in and out of humidity sensing mode by toggling wall switch\nENERGY STAR qualified for efficient cost-saving operation\nPrecision engineered with DC brushless motor for extended reliability, this fan will outlast many household appliances",DELTA ELECTRONICS (AMERICAS) LTD.,White,us
167169,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed Fan/Light Bathroom Ventilation Fan with White Trim Ring,NaN,"Super quiet 80CFM energy efficient fan virtually disappears into the ceiling leaving only a recessed light in view\nMay be installed over shower when wired to a GFCI breaker and used with a PAR30L 75W (max) CFL\nBulb not included. Accepts any of the following bulbs: 75W Max. PAR30, 14W Max. BR30 LED, or 75W Max. PAR30L (for use over tub/shower.)",Aero Pure,White,us
167170,B011RX6PNO,"Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroom Fan with LED Light and Humidity Sensor, White Finish",NaN,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joists Or Greater\nCountry of Origin: China\nBrand name: Aero Pure\nItem Dimensions: 11. 38""L x 10. 5""W x 5. 88""H",Aero Pure,White Finish,us
167171,B01MZIK0PI,"Delta Electronics (Americas) Ltd. RAD80 Delta BreezRadiance Series 80 CFM Fan with Heater, 10.5W, 1.5 Sones",NaN,"Quiet operation at 1.5 Sones\nPrecision engineered with DC brushless motor for extended reliability, this Fan will outlast many household appliances\nGalvanized steel construction resists corrosion, equipped with metal duct adapter\nFan impeller Stops If obstructed, for safe worry-free operation\nPeace of mind quality, performance and reliability from the world's largest DC brushless Fan Manufacturer",DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us
167172,B01N5Y6002,"Delta Electronics (Americas) Ltd. GBR80HLED Delta BreezGreenBuilder Series 80 CFM Fan/Dimmable H, LED Light, Dual Speed & Humidity Sensor",NaN,"Ultra energy-efficient LED module (11-watt equivalent to 60-watt incandescent light) included. Main light output-850 Lumens, 3000K\nExtracts air at a rate of 80 CFM to properly ventilate bathrooms up to 80 sq. Ft., quiet operation at 0.8 sones\nPrecision engineered with DC brushless motor for extended reliability, this Fan will outlast many household appliances\nEnergy Star qualified for efficient cost-saving operation, galvanized steel construction resists corrosion\nFan impeller Stops If obstructed, for safe worry-free operation, attractive grille gives your bathroom a fresh look",DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us
167173,B07JY1PQNT,"Aero Pure ABF80 L5 W ABF80L5 Ceiling Mount 80 CFM w/LED Light/Nightlight, Energy Star Certified, White Quiet Bathroom Ventilation Fan",NaN,"Quiet 0.3 Sones, 80 CFM fan with choice of three designer grilles in White, Satin Nickel, or Oil Rubbed Bronze; Full 6 year warranty\n10W 3000K 800 Lumens LED Light with 0.7W Nightlight included\nInstallation friendly- Quick-mount adjustable metal bracket for new construction and retrofit; 4”, 5: and 6” metal duct adaptor included\nMeets today’s demanding building specifications- ETL Listed for wet application, ENERGY STAR certified, CALGreen, JA-8 Compliant for CA Title 24, and ASHRAE 62.2 compliant\nHousing dimensions- 10 2/5”x10 2/5”x 7 ½”; Grille dimensions- 13”x13”; Fits 2""x8"" joists",Aero Pure,White,us
167174,B07QJ7WYFQ,Panasonic FV-08VRE2 Ventilation Fan with Recessed LED (Renewed),NaN,"The design solution for Fan/light combinations\nEnergy Star rated architectural grade recessed Fan/LED light\nQuiet, energy efficient and powerful 80 CFM ventilation hidden above the Ceiling\nLED lamp is dimmable\nBeautiful Lighting with 6-1/2”aperture and advanced luminaire design",Pa

In [49]:
examples[examples['product_locale'] == 'us'].head(100)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train
5,5,revent 80 cfm,0,B076Q7V5WX,us,E,0,1,train
6,6,revent 80 cfm,0,B075ZBF9HG,us,E,0,1,train
7,7,revent 80 cfm,0,B06W2LB17J,us,E,0,1,train
8,8,revent 80 cfm,0,B07JY1PQNT,us,E,0,1,train
9,9,revent 80 cfm,0,B01MZIK0PI,us,E,0,1,train


In [50]:
products['product_locale'].unique()

<ArrowStringArray>
['es', 'us', 'jp']
Length: 3, dtype: str

In [51]:
products[products['product_locale'] == 'us'].shape

(1215854, 7)

In [52]:
products.shape

(1814924, 7)

In [53]:
examples[
    examples["product_locale"] == "us"
].head(100)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train
5,5,revent 80 cfm,0,B076Q7V5WX,us,E,0,1,train
6,6,revent 80 cfm,0,B075ZBF9HG,us,E,0,1,train
7,7,revent 80 cfm,0,B06W2LB17J,us,E,0,1,train
8,8,revent 80 cfm,0,B07JY1PQNT,us,E,0,1,train
9,9,revent 80 cfm,0,B01MZIK0PI,us,E,0,1,train


In [54]:
examples.shape

(2621288, 9)

In [55]:
products['product_locale'].value_counts()

product_locale
us    1215854
jp     339059
es     260011
Name: count, dtype: int64

In [56]:
# is product_id unique, or does the same product repeat across locales?
products['product_id'].duplicated().sum()

np.int64(12152)

In [57]:
us_products = products[products['product_locale'] == 'us']
us_products['product_id'].duplicated().sum()

np.int64(0)

In [58]:
us_examples = examples[examples['product_locale'] == 'us']

In [59]:
# esci_label distribution
examples['esci_label'].value_counts()

esci_label
E    1708158
S     574313
I     263165
C      75652
Name: count, dtype: int64

In [60]:
# small vs large version split sizes
examples[['small_version','large_version']].sum()

small_version    1118011
large_version    2621288
dtype: int64

In [61]:
group = us_examples.groupby("query_id")
flags = group[["split", "small_version"]].nunique()
flags["n_texts"] = group["query"].nunique()
bad_ids = flags[(flags["split"] > 1) | (flags["small_version"] > 1) | (flags["n_texts"] > 1)].index

In [64]:
print(flags.loc[bad_ids].to_string())

          split  small_version  n_texts
query_id                               
694           1              2        1
26525         1              2        1
46791         1              2        1
79706         2              2        1
91973         1              2        1
98413         1              2        1
114163        1              2        1


In [62]:
print(len(bad_ids))

7


In [5]:
query_words = examples["query"].str.split().str.len()

query_words.describe()

count    2.621288e+06
mean     3.253567e+00
std      1.718152e+00
min      1.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      4.000000e+00
max      2.900000e+01
Name: query, dtype: float64

In [63]:
nlqec_ids = sources.loc[
    sources["source"] == "nlqec", "query_id"
]

nlqec_queries = examples[
    examples["query_id"].isin(nlqec_ids)
][["query_id", "query"]].drop_duplicates()

display(nlqec_queries)

,query_id,query
57831,2162,15 inch light weight laptop that has lots of memory storage and RAM. It should include a HDMI port as well as a USB drive.
133318,5608,A Lenovo like mine its a little notebook it's fab. It would be black in colour
133350,5609,A Windows laptop that I can use to play and stream games. It should have at least a 15 screen.
133366,5610,"A basic, denim jacket that has a slightly oversized fit."
133405,5611,"A bomber jacket with inner warming lining, preferrably black"
133445,5612,A laptop that had good gaming capabilities while being budget friendly
133460,5613,A laptop that is light weight and with high specifications which I could use for working when at home or when travelling.
133500,5614,A laptop which runs faster and has more capacity to run downloaded software
133516,5615,A laptop with enough power to play 4K videos and run photoshop easily.
133531,5616,A lightweight one with long battery life and quick processor with adequate storage. One that can be transported easily with at least a 13 inch screen and backlit keyboard.
